<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

# PPE Monitor — اجرای سریع

فقط سه سلول:

| سلول | کار | زمان |
|---|---|---|
| ۱ | اتصال Drive + نصب + بارگذاری مدل‌ها | **بار اول:** حدود ۱۰ دقیقه (دانلود وزن‌ها و ساخت موتورهای TensorRT، که روی Drive ذخیره می‌شوند)<br>**دفعه‌های بعد:** حدود ۱ دقیقه |
| ۲ | تحلیل یک ویدئو یا عکس، فقط با دادن آدرس | — |
| ۳ | (اختیاری) آپلود فایل از کامپیوتر و تحلیل آن | — |

**پوشه‌ی Drive:** `MyDrive/ppe_monitor/`

```text
ppe_monitor/
├── ppe_monitor.py      ← کد (از GitHub دانلود می‌شود، یا بار اول آپلودش کن)
├── weights/            ← وزن‌های .pt (خودکار)
├── engines/Tesla_T4/   ← موتورهای TensorRT (خودکار، برای هر نوع GPU جدا)
├── inputs/             ← ویدئو و عکس‌هایت را این‌جا بگذار
└── outputs/            ← خروجی‌ها + events/ (عکس رخدادهای تخلف)
```

Runtime → Change runtime type → **T4 GPU** را فراموش نکن.

</div>

In [ ]:
# ============================================================
# ۱) راه‌اندازی — هر session یک بار
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
!pip -q install ultralytics "lap>=0.5.12" arabic-reshaper python-bidi

import os, sys, urllib.request
BASE = "/content/drive/MyDrive/ppe_monitor"
os.makedirs(BASE, exist_ok=True)

# بعد از ساختن ریپوی GitHub، آدرس raw فایل را این‌جا بگذار تا همیشه آخرین نسخه گرفته شود:
MODULE_URL = ""   # مثلاً "https://raw.githubusercontent.com/<username>/ppe-monitor/main/ppe_monitor.py"
if MODULE_URL:
    urllib.request.urlretrieve(MODULE_URL, f"{BASE}/ppe_monitor.py")
if not os.path.exists(f"{BASE}/ppe_monitor.py"):
    from google.colab import files
    print("فایل ppe_monitor.py را انتخاب کن (فقط بار اول):")
    for name, data in files.upload().items():
        open(f"{BASE}/ppe_monitor.py", "wb").write(data)

sys.path.insert(0, BASE)
import ppe_monitor as pm
pm.init()          # پارامترها را هم می‌شود داد، مثلاً: pm.init(backend="pytorch", show_skeleton=False)

In [ ]:
# ============================================================
# ۲) تحلیل — فقط آدرس بده (نام خالی = از پوشه‌ی inputs/ روی Drive)
# ============================================================
result = pm.analyze("3.mp4")

# نمونه‌های دیگر:
# pm.analyze("/content/drive/MyDrive/videos/site_cam1.mp4")
# pm.analyze("photo.jpg")
# pm.analyze("3.mp4", debug=True)          # تشخیص‌های ردشده با دلیلشان
# pm.analyze("long.mp4", max_seconds=30)   # فقط ۳۰ ثانیه‌ی اول
# pm.analyze("3.mp4", benchmark=True)      # + مقایسه‌ی سرعت و دقت TensorRT و PyTorch

In [ ]:
# ============================================================
# ۳) (اختیاری) آپلود از کامپیوتر و تحلیل
# ============================================================
from google.colab import files
for name, data in files.upload().items():
    path = pm.INPUT_DIR / name
    path.write_bytes(data)
    pm.analyze(path)

<div style="direction:rtl;text-align:right;font-family:B Lotus, B Nazanin, Tahoma">

## تنظیمات رایج

```python
pm.cfg.show_skeleton = False      # بدون اسکلت (بعدش pm.init() را دوباره اجرا کن)
pm.cfg.person_conf = 0.15         # آدم‌های دورتر
pm.cfg.on_violation = 2.0         # هشدار محتاط‌تر
pm.cfg.required_ppe = ("helmet",) # فقط کلاه اجباری است
```

همه‌ی پارامترها و راهنمای «کدام علامت → کدام پارامتر» در `README.md` ریپو آمده است. اگر TensorRT مشکل داشت، `pm.init(backend="pytorch")` دقیقاً رفتار V2 را دارد.

</div>